<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/RAG_with_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with LangChain

## Goal

Build a simple **Retrieval-Augmented Generation (RAG)** question-answering system using:

- the public dataset `m-ric/huggingface_doc`;
- LangChain `Document` objects;
- `RecursiveCharacterTextSplitter`;
- `sentence-transformers/all-MiniLM-L6-v2` embeddings;
- a FAISS vector store;
- a retriever with different values of `k`;
- the local model `google/flan-t5-small`;
- a `RetrievalQA` chain that returns both answers and source documents.

No API key is required.

In [ ]:
# Install the libraries required by the exercise.
# Restart the runtime only if Colab explicitly asks you to do so.

%pip install -q -U \
    "datasets<5" \
    "transformers<5" \
    sentence-transformers \
    faiss-cpu \
    accelerate \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-classic

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else -1
print("PyTorch version:", torch.__version__)
print("Device:", "GPU" if DEVICE == 0 else "CPU")

## 1. Load the dataset

We load only the first 200 rows to keep the exercise fast. Then we inspect the available columns and one raw example.

In [ ]:
dataset = load_dataset(
    "m-ric/huggingface_doc",
    split="train[:200]"
)

print("Number of rows:", len(dataset))
print("Columns:", dataset.column_names)

example = dataset[0]
print("\nExample source:")
print(example["source"])

print("\nExample text preview:")
print(example["text"][:1000])

## 2. Convert rows into LangChain Documents

Each row becomes a LangChain `Document`.

- `page_content` contains the documentation text.
- `metadata["source"]` stores the original source.
- `metadata["row_id"]` helps trace a chunk back to its dataset row.

In [ ]:
documents = []

for row_id, row in enumerate(dataset):
    text = str(row.get("text", "")).strip()
    source = str(row.get("source", "unknown_source")).strip()

    if not text:
        continue

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": source,
                "row_id": row_id,
            },
        )
    )

print("LangChain documents created:", len(documents))
print("\nFirst document metadata:", documents[0].metadata)
print("\nFirst document preview:")
print(documents[0].page_content[:500])

## 3. Chunk the documents using two strategies

Two configurations are compared:

| Strategy | `chunk_size` | `chunk_overlap` | Main purpose |
|---|---:|---:|---|
| A | 450 | 60 | More precise and compact chunks |
| B | 800 | 120 | More surrounding context per chunk |

Smaller chunks can improve precision, while larger chunks preserve more context. Overlap reduces the risk of losing information at chunk boundaries.

In [ ]:
def create_chunks(source_documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        add_start_index=True,
    )
    return splitter.split_documents(source_documents)


chunks_a = create_chunks(
    documents,
    chunk_size=450,
    chunk_overlap=60,
)

chunks_b = create_chunks(
    documents,
    chunk_size=800,
    chunk_overlap=120,
)


def chunk_statistics(name, chunks):
    lengths = [len(chunk.page_content) for chunk in chunks]
    return {
        "strategy": name,
        "number_of_chunks": len(chunks),
        "average_length": round(float(np.mean(lengths)), 2),
        "minimum_length": int(np.min(lengths)),
        "maximum_length": int(np.max(lengths)),
    }


chunk_comparison = pd.DataFrame([
    chunk_statistics("A: 450 / 60", chunks_a),
    chunk_statistics("B: 800 / 120", chunks_b),
])

display(chunk_comparison)

print("\nStrategy A - first chunk:")
print(chunks_a[0].page_content[:600])
print("Metadata:", chunks_a[0].metadata)

print("\nStrategy B - first chunk:")
print(chunks_b[0].page_content[:900])
print("Metadata:", chunks_b[0].metadata)

## 4. Build FAISS vector stores and retrievers

The embedding model converts every chunk into a numerical vector. FAISS stores those vectors and performs similarity search.

We first build one vector store for each chunking strategy so their retrieval results can be compared fairly.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

vector_store_a = FAISS.from_documents(chunks_a, embeddings)
vector_store_b = FAISS.from_documents(chunks_b, embeddings)

retriever_a = vector_store_a.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retriever_b = vector_store_b.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("The two FAISS vector stores are ready.")

## 5. Compare the two chunking strategies

The same question is sent to both retrievers. Reading the returned chunks is essential because a RAG system cannot generate a grounded answer when retrieval is poor.

In [ ]:
SANITY_CHECK_QUESTION = (
    "How can a user load a dataset with the Hugging Face Datasets library?"
)


def retrieve_documents(retriever, question):
    return retriever.invoke(question)


def print_retrieved_chunks(title, retrieved_documents):
    print("=" * 100)
    print(title)
    print("=" * 100)

    for rank, doc in enumerate(retrieved_documents, start=1):
        source = doc.metadata.get("source", "unknown_source")
        row_id = doc.metadata.get("row_id", "unknown")
        start_index = doc.metadata.get("start_index", "unknown")

        print(f"\nRank {rank}")
        print(f"Source: {source}")
        print(f"Row ID: {row_id} | Start index: {start_index}")
        print("Content:")
        print(doc.page_content[:900].replace("\n", " "))


retrieved_a = retrieve_documents(retriever_a, SANITY_CHECK_QUESTION)
retrieved_b = retrieve_documents(retriever_b, SANITY_CHECK_QUESTION)

print("Question:", SANITY_CHECK_QUESTION)
print_retrieved_chunks(
    "Strategy A — chunk_size=450, chunk_overlap=60",
    retrieved_a,
)
print_retrieved_chunks(
    "Strategy B — chunk_size=800, chunk_overlap=120",
    retrieved_b,
)

### Chunking interpretation

- Strategy A usually returns shorter passages focused on a narrower idea.
- Strategy B usually provides more surrounding context, but some passages may contain extra information unrelated to the question.
- For `flan-t5-small`, compact chunks are helpful because the model has a limited input context.
- Therefore, **Strategy A (`450 / 60`)** is selected for the final RAG pipeline.

The code output above must still be inspected. If its chunks are not relevant, change the question or test other chunk sizes before generating an answer.

## 6. Retrieval sanity check with `k = 2`, `k = 4`, and `k = 6`

The value of `k` controls how many chunks are returned:

- `k=2`: less context and usually higher precision;
- `k=4`: a balance between precision and coverage;
- `k=6`: broader coverage, but more noise and a longer prompt.

We test all three values before selecting the final retriever.

In [ ]:
def evaluate_k_values(vector_store, question, k_values=(2, 4, 6)):
    rows = []

    for k in k_values:
        retriever = vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k},
        )
        retrieved = retrieve_documents(retriever, question)

        unique_sources = []
        for doc in retrieved:
            source = doc.metadata.get("source", "unknown_source")
            if source not in unique_sources:
                unique_sources.append(source)

        rows.append({
            "k": k,
            "chunks_returned": len(retrieved),
            "unique_sources": len(unique_sources),
            "sources": " | ".join(unique_sources),
        })

        print_retrieved_chunks(
            f"Retrieval results for k={k}",
            retrieved,
        )

    return pd.DataFrame(rows)


k_comparison = evaluate_k_values(
    vector_store_a,
    SANITY_CHECK_QUESTION,
    k_values=(2, 4, 6),
)

print("\nSummary:")
display(k_comparison)

### Retrieval decision

`k=4` is used for the final system because it generally gives enough evidence without making the prompt unnecessarily long. This is especially important with a small local language model.

A final retriever is now created from Strategy A with `k=4`.

In [ ]:
FINAL_K = 4

final_retriever = vector_store_a.as_retriever(
    search_type="similarity",
    search_kwargs={"k": FINAL_K},
)

final_sanity_chunks = retrieve_documents(
    final_retriever,
    SANITY_CHECK_QUESTION,
)

print_retrieved_chunks(
    "FINAL RETRIEVAL SANITY CHECK",
    final_sanity_chunks,
)

## 7. Load the local FLAN-T5 model

`google/flan-t5-small` is an instruction-tuned sequence-to-sequence model. It runs locally after download and does not require an API key.

In [ ]:
GENERATION_MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL_NAME
)

text_generation_pipeline = pipeline(
    task="text2text-generation",
    model=generation_model,
    tokenizer=tokenizer,
    device=DEVICE,
    max_new_tokens=160,
    do_sample=False,
    truncation=True,
)

local_llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)

print("Local language model loaded:", GENERATION_MODEL_NAME)

## 8. Build the RetrievalQA chain

The prompt explicitly tells the model to use only the retrieved context. When the answer is absent from the context, the model should say that it does not know.

In [ ]:
RAG_PROMPT_TEMPLATE = """
You are a question-answering assistant.

Use only the context below to answer the question.
Do not use outside knowledge.
If the answer is not present in the context, say:
"I do not know based on the retrieved context."

Give a concise and factual answer.

Context:
{context}

Question:
{question}

Answer:
""".strip()

rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

rag_chain = RetrievalQA.from_chain_type(
    llm=local_llm,
    chain_type="stuff",
    retriever=final_retriever,
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": rag_prompt,
    },
)

print("RetrievalQA chain created successfully.")

## 9. Ask questions and print answers with sources

For every question, the notebook prints:

1. the generated answer;
2. the sources used by the retriever;
3. a preview of each retrieved chunk.

This makes it possible to verify whether the answer is grounded in the dataset.

In [ ]:
def ask_rag(question, show_chunk_previews=True):
    response = rag_chain.invoke({"query": question})

    answer = response.get("result", "").strip()
    source_documents = response.get("source_documents", [])

    print("=" * 100)
    print("QUESTION")
    print(question)

    print("\nANSWER")
    print(answer)

    print("\nRETRIEVED SOURCES")
    if not source_documents:
        print("No source document was returned.")
        return response

    seen_sources = set()

    for index, doc in enumerate(source_documents, start=1):
        source = doc.metadata.get("source", "unknown_source")
        row_id = doc.metadata.get("row_id", "unknown")
        start_index = doc.metadata.get("start_index", "unknown")

        print(f"\n[{index}] {source}")
        print(f"Row ID: {row_id} | Start index: {start_index}")

        if show_chunk_previews:
            preview = doc.page_content[:700].replace("\n", " ")
            print("Chunk preview:", preview)

        seen_sources.add(source)

    print("\nNumber of retrieved chunks:", len(source_documents))
    print("Number of unique sources:", len(seen_sources))

    return response


questions = [
    (
        "How can a user load a dataset with the "
        "Hugging Face Datasets library?"
    ),
    "What is the purpose of the Trainer class?",
    (
        "What does a pipeline help a user do in "
        "the Transformers library?"
    ),
]

results = {}

for question in questions:
    results[question] = ask_rag(question)
    print("\n")

## 10. Final analysis

### What is a vector store?

A vector store keeps numerical representations of text chunks. It allows the system to search for chunks that are semantically similar to a user's question rather than relying only on exact keyword matches.

### Why is the retriever essential?

The retriever selects the most relevant chunks from the vector store. Those chunks become the context supplied to the language model. Good retrieval reduces hallucination and makes answers traceable to sources.

### Why must documents be chunked?

Whole documents can be too long and may contain several unrelated topics. Chunking creates smaller searchable units, improves retrieval precision, and keeps the generated prompt within the model's context limit.

### Effect of chunk size and overlap

- Smaller chunks are more focused but can lose surrounding explanations.
- Larger chunks preserve context but may introduce irrelevant content.
- Overlap protects information that crosses a chunk boundary.
- Excessive overlap creates redundant chunks and increases storage and computation.

### Effect of `k`

- A low `k` may miss useful evidence.
- A high `k` may add noise and exceed the language model's context capacity.
- `k=4` is a reasonable compromise for this small experiment, but the best value depends on the dataset, questions, chunking strategy, and model.

### Debugging approach

Before trusting a generated answer:

1. inspect the retrieved chunks;
2. verify their source metadata;
3. check whether the answer is explicitly supported;
4. adjust `chunk_size`, `chunk_overlap`, or `k` when retrieval is weak;
5. test with several questions, including one whose answer is not present.

The retrieved chunks are evidence. A fluent answer without relevant evidence should not be considered reliable.

## Optional negative test

Use a question that is unlikely to be answered by Hugging Face documentation. The model should respond that it does not know based on the retrieved context.

In [ ]:
negative_test_question = (
    "What was the final score of yesterday's football match?"
)

negative_test_result = ask_rag(
    negative_test_question,
    show_chunk_previews=True,
)